# 🚀 NCKH 3 — Kaggle Thin Runner for GPT2SP (GitHub ➔ Kaggle ➔ Hugging Face Hub)

Notebook này đóng vai trò là **Thin Runner** siêu gọn nhẹ chạy trên Kaggle GPU/CPU:
1. **GitHub** lưu toàn bộ mã nguồn Python sạch trên từng branch thử nghiệm.
2. **Kaggle Runtime** chỉ làm nhiệm vụ đọc Kaggle Secrets (`hf_token` / `HF_TOKEN`), `git clone` đúng branch và chạy script thực nghiệm.
3. **Hugging Face Hub** tự động nhận toàn bộ kết quả (`.csv`, `.png`, `run_metadata.json`) vào **Dataset Repo** và model checkpoints (`.bin`) vào **Model Repo** ngay trong khi chạy (kèm Auto-Resume nếu mất mạng).

## ⚙️ Cell 1: Cấu hình Thực nghiệm (Chỉnh sửa tham số tại đây)

In [ ]:
import os
import shlex
import subprocess

# ==============================================================================
# CHỈNH SỬA CẤU HÌNH CHO MỖI LẦN CHẠY TẠI ĐÂY
# ==============================================================================
GITHUB_REPO = "https://github.com/snowflake-vibing/NCKH_Johnny_Snowflake.git"
GIT_BRANCH = "main"  # Đổi thành tên branch nếu chạy thử nghiệm trên nhánh riêng

# Chọn 1 trong 2 thực nghiệm GPT2SP: "gpt2sp_16projects", "gpt2sp_retrain"
EXPERIMENT = "gpt2sp_16projects"

HF_RESULTS_REPO = "NCKH_GPT2SP"
HF_CHECKPOINTS_REPO = "nckh3-gpt2sp-checkpoints"

# Tham số dòng lệnh bổ sung (vd: "--projects springxd,bamboo" hoặc "--setting full_128 --epochs 20")
EXTRA_ARGS = ""

print(f"Repo: {GITHUB_REPO} | Branch: {GIT_BRANCH} | Experiment: {EXPERIMENT}")

## 🔐 Cell 2: Kết nối Kaggle Secrets & Xác thực Hugging Face (Write Token)

In [ ]:
from huggingface_hub import HfApi

def load_hf_token_silently():
    for key in ("HF_TOKEN", "hf_token", "HUGGINGFACE_HUB_TOKEN"):
        val = os.environ.get(key)
        if val and val.strip():
            return val.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for key in ("HF_TOKEN", "hf_token"):
            try:
                val = secrets.get_secret(key)
                if val and val.strip():
                    return val.strip()
            except Exception:
                continue
    except ImportError:
        pass
    return None

token = load_hf_token_silently()
if not token:
    raise RuntimeError(
        "Không tìm thấy HF_TOKEN hoặc hf_token trong Kaggle Secrets! "
        "Hãy vào Add-ons -> Secrets và bật token có quyền WRITE."
    )

os.environ["HF_TOKEN"] = token
api = HfApi(token=token)
HF_USERNAME = api.whoami()["name"]
print(f"✅ Đã xác thực Hugging Face thành công với tài khoản: {HF_USERNAME}")

## 📥 Cell 3: Clone / Cập nhật Mã nguồn từ GitHub Branch

In [ ]:
WORK_ROOT = "/kaggle/working" if os.path.exists("/kaggle/working") else os.getcwd()
REPO_DIR = os.path.join(WORK_ROOT, "repo")

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--branch", GIT_BRANCH, "--single-branch", GITHUB_REPO, REPO_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", GIT_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", GIT_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", GIT_BRANCH], check=True)

os.chdir(REPO_DIR)
commit_hash = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()

import torch
gpu_desc = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"✅ Đã sẵn sàng tại {REPO_DIR} | Branch: {GIT_BRANCH} | Commit: {commit_hash} | Hardware: {gpu_desc}")

## ▶️ Cell 4: Cài đặt Thư viện Bổ sung (nếu cần) & Chạy Script Thực nghiệm

In [ ]:
import importlib.util
import sys

REQUIRED_PKGS = ["huggingface_hub", "transformers", "scipy", "tqdm"]
missing_pkgs = [pkg for pkg in REQUIRED_PKGS if importlib.util.find_spec(pkg) is None]
if missing_pkgs:
    print("📦 Installing missing packages:", missing_pkgs)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs], check=True)

EXPERIMENT_SCRIPTS = {
    "gpt2sp_16projects": "kaggle_GPT2SP/run_gpt2sp_16projects.py",
    "gpt2sp_retrain": "kaggle_GPT2SP/run_gpt2sp_retrain.py",
}

if EXPERIMENT not in EXPERIMENT_SCRIPTS:
    raise ValueError(f"EXPERIMENT không hợp lệ: {EXPERIMENT}. Chọn: {list(EXPERIMENT_SCRIPTS.keys())}")

cmd = [sys.executable, "-u", EXPERIMENT_SCRIPTS[EXPERIMENT], "--results-repo", HF_RESULTS_REPO]
if EXPERIMENT == "gpt2sp_retrain":
    cmd.extend(["--checkpoints-repo", HF_CHECKPOINTS_REPO])
if EXTRA_ARGS.strip():
    cmd.extend(shlex.split(EXTRA_ARGS))

print("🚀 Executing:", " ".join(cmd))
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"Script exited with code {ret}")

## 📊 Cell 5: Hiển thị Tóm tắt Kết quả & Liên kết Hugging Face Hub

In [ ]:
import glob
import pandas as pd

outputs_root = os.path.join(WORK_ROOT, "outputs")
csv_files = sorted(glob.glob(os.path.join(outputs_root, "**", "*.csv"), recursive=True))
summary_files = [p for p in csv_files if "summary" in os.path.basename(p) or "audit" in os.path.basename(p)]

print(f"📦 Hugging Face Results Dataset: https://huggingface.co/datasets/{HF_USERNAME}/{HF_RESULTS_REPO}")
if EXPERIMENT == "gpt2sp_retrain":
    print(f"🧠 Hugging Face Checkpoints Model: https://huggingface.co/{HF_USERNAME}/{HF_CHECKPOINTS_REPO}")

if summary_files:
    latest_summary = summary_files[-1]
    print(f"\n📄 Summary table ({latest_summary}):")
    df_sum = pd.read_csv(latest_summary)
    try:
        display(df_sum)
    except NameError:
        print(df_sum.to_string(index=False))
else:
    print("ℹ️ Không tìm thấy file summary CSV cục bộ; kiểm tra trực tiếp trên HF Dataset Repo.")